In [0]:

# Define project paths
base_path = "/Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01"

bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"

# Define the catalog and schema
catalog_name = "adv_dev_westus3_demo_001"
schema_name = "ecommerce_01"

# Display paths
print("Bronze path:", bronze_path)
print("Silver path:", silver_path)
print("Target schema:", f"{catalog_name}.{schema_name}")


Bronze path: /Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/bronze
Silver path: /Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/silver
Target schema: adv_dev_westus3_demo_001.ecommerce_01


In [0]:
# Create and read the dataset parameter
dbutils.widgets.text("dataset_name", "ALL", "Dataset name")

dataset_name = dbutils.widgets.get("dataset_name").strip().lower()

print("Selected dataset:", dataset_name)

Selected dataset: all


In [0]:

from pyspark.sql.functions import col

# Reusable function for dynamic data cleaning
def clean_data(df, key_columns):
    # Remove duplicate records
    df = df.dropDuplicates(key_columns)

    # Remove rows with missing key values
    for column in key_columns:
        df = df.filter(col(column).isNotNull())

    return df

print("Dynamic cleaning function is ready.")


Dynamic cleaning function is ready.


In [0]:

# Define datasets and their unique key columns
datasets = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "order_payments": ["order_id", "payment_sequential"]
}

# Select datasets using the parameter
if dataset_name == "all":
    selected_datasets = datasets
else:
    if dataset_name not in datasets:
        raise ValueError(f"Invalid dataset name: {dataset_name}")
    selected_datasets = {dataset_name: datasets[dataset_name]}

# Clean and save each selected dataset
for name, keys in selected_datasets.items():
    print(f"Processing {name}...")

    df = spark.read.parquet(f"{bronze_path}/{name}")

    cleaned_df = clean_data(df, keys)

    table_name = f"silver_{name}"

    cleaned_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")

    print(f"Saved {table_name} successfully.")
    
print("Silver processing completed.")


Processing orders...
Saved silver_orders successfully.
Processing order_items...
Saved silver_order_items successfully.
Processing customers...
Saved silver_customers successfully.
Processing products...
Saved silver_products successfully.
Processing order_payments...
Saved silver_order_payments successfully.
Silver processing completed.


In [0]:

# Verify the Silver Delta tables
for name in [
    "silver_orders",
    "silver_order_items",
    "silver_customers",
    "silver_products",
    "silver_order_payments"
]:
    table_name = f"{catalog_name}.{schema_name}.{name}"
    print(f"{table_name}: {spark.table(table_name).count()} rows")


adv_dev_westus3_demo_001.ecommerce_01.silver_orders: 99441 rows
adv_dev_westus3_demo_001.ecommerce_01.silver_order_items: 112650 rows
adv_dev_westus3_demo_001.ecommerce_01.silver_customers: 99441 rows
adv_dev_westus3_demo_001.ecommerce_01.silver_products: 32951 rows
adv_dev_westus3_demo_001.ecommerce_01.silver_order_payments: 103886 rows
